# Parametric PINNs and Inverse Problems

**Time: ~40 minutes**

Two of the most powerful PINN applications:

1. **Parametric PINNs** — solve an entire family of equations with one model
2. **Inverse PINNs** — infer unknown physical parameters from data

Both build on everything from notebooks 01-06.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

---
## Part 1: Parametric PINNs

### The Idea

Standard PINN: one model solves **one** equation instance.

Parametric PINN: parameters become **network inputs**. The model learns the solution `u(t; k)` for a whole range of `k`.

### Example: `u' = -k*u`, `u(0) = 1`

Solution: `u(t; k) = exp(-k*t)` for any `k > 0`.

In [ ]:
# Parametric model: takes (t, k) as input
class ParametricPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(),  # 2 inputs: t, k
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, t, k):
        return self.net(torch.cat([t, k], dim=1))

model = ParametricPINN()
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# Training: sample (t, k) pairs randomly
N = 2000
t_train = torch.rand(N, 1).requires_grad_(True) * 3  # t in [0, 3]
k_train = torch.rand(N, 1) * 4 + 0.5                 # k in [0.5, 4.5]
k_train.requires_grad_(False)

# IC points: t=0, various k
N_ic = 200
k_ic = torch.rand(N_ic, 1) * 4 + 0.5
t_ic = torch.zeros(N_ic, 1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(8000):
    optimizer.zero_grad()

    # Physics loss: u' + k*u = 0
    u = model(t_train, k_train)
    du_dt = torch.autograd.grad(u, t_train, torch.ones_like(u), create_graph=True)[0]
    residual = du_dt + k_train * u
    loss_phys = torch.mean(residual**2)

    # IC loss: u(0, k) = 1
    loss_ic = torch.mean((model(t_ic, k_ic) - 1.0)**2)

    loss = loss_phys + 20.0 * loss_ic
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 2000 == 0:
        print(f"Epoch {epoch:5d} | phys={loss_phys.item():.3e} | ic={loss_ic.item():.3e}")

print(f"Final loss: {losses[-1]:.4e}")

In [ ]:
# Evaluate: one model, multiple k values — including values NOT seen during training
t_test = torch.linspace(0, 3, 200).unsqueeze(1)

fig, ax = plt.subplots(figsize=(10, 6))
for k_val in [0.5, 1.0, 2.0, 3.0, 4.5]:
    k_tensor = torch.full_like(t_test, k_val)
    with torch.no_grad():
        u_pred = model(t_test, k_tensor).numpy()
    u_exact = np.exp(-k_val * t_test.numpy())
    ax.plot(t_test.numpy(), u_exact, 'k-', alpha=0.4, linewidth=1)
    ax.plot(t_test.numpy(), u_pred, '--', linewidth=2, label=f'k = {k_val}')

ax.set_xlabel('t'); ax.set_ylabel('u(t; k)')
ax.set_title('Parametric PINN: one model, whole family of solutions')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Black lines = exact. Dashed = PINN predictions.")
print("A single model learned the entire solution family u(t; k) = exp(-kt).")

### Design Rules for Parametric PINNs

1. **Normalize parameter inputs** to `[-1, 1]` — prevents gradient imbalance
2. **Normalize the residual** by the dominant scaling — keeps loss O(1)
3. **Embed known structure** analytically (Ansatz) — let the network learn the easy part
4. **Validate on held-out parameters** — interpolation ≠ extrapolation

See `experiments/parametric_harmonic/` and `docs/parametric_pinns.md` in this repo for a production implementation.

---
## Part 2: Inverse Problems

### The Idea

We **observe** the solution at some points, but we **don't know** a parameter in the equation. Can we infer it?

This flips the PINN: instead of solving a known equation, we **learn the equation** from data.

### Example: `u' = -k*u`, k is unknown

We have noisy measurements of `u(t)` and want to find `k`.

In [ ]:
# Generate synthetic "observed" data with true k = 2.0
k_true = 2.0
np.random.seed(42)

t_obs_np = np.sort(np.random.uniform(0, 2, 30))
u_obs_np = np.exp(-k_true * t_obs_np) + 0.02 * np.random.randn(30)

t_obs = torch.tensor(t_obs_np, dtype=torch.float32).unsqueeze(1)
u_obs = torch.tensor(u_obs_np, dtype=torch.float32).unsqueeze(1)

plt.figure(figsize=(8, 4))
t_fine = np.linspace(0, 2, 200)
plt.plot(t_fine, np.exp(-k_true * t_fine), 'b-', label=f'True: exp(-{k_true}t)')
plt.scatter(t_obs_np, u_obs_np, c='red', s=30, label='Observed (noisy)')
plt.xlabel('t'); plt.ylabel('u(t)')
plt.title(f'Inverse problem: find k from noisy data (true k = {k_true})')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
class InversePINN(nn.Module):
    """PINN with a learnable parameter k."""
    def __init__(self, k_init=1.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 1),
        )
        # Learnable parameter — optimized jointly with the network weights
        self.log_k = nn.Parameter(torch.tensor(np.log(k_init)))

    @property
    def k(self):
        return torch.exp(self.log_k)  # ensure k > 0

    def forward(self, t):
        return self.net(t)

model_inv = InversePINN(k_init=0.5)  # Start with wrong initial guess
print(f"Initial k guess: {model_inv.k.item():.4f} (true: {k_true})")

In [ ]:
# Physics collocation points
t_phys = torch.linspace(0, 2, 200).unsqueeze(1).requires_grad_(True)

optimizer = torch.optim.Adam(model_inv.parameters(), lr=1e-3)
k_history = []
loss_history = []

for epoch in range(10000):
    optimizer.zero_grad()

    # Data loss: match observations
    loss_data = torch.mean((model_inv(t_obs) - u_obs)**2)

    # Physics loss: u' + k*u = 0 (k is learnable!)
    u = model_inv(t_phys)
    du_dt = torch.autograd.grad(u, t_phys, torch.ones_like(u), create_graph=True)[0]
    residual = du_dt + model_inv.k * u
    loss_phys = torch.mean(residual**2)

    # IC loss
    loss_ic = (model_inv(torch.zeros(1, 1)) - 1.0)**2

    loss = loss_data + loss_phys + 10 * loss_ic
    loss.backward()
    optimizer.step()

    k_history.append(model_inv.k.item())
    loss_history.append(loss.item())

    if epoch % 2000 == 0:
        print(f"Epoch {epoch:5d} | k = {model_inv.k.item():.4f} | loss = {loss.item():.4e}")

print(f"\nFinal k = {model_inv.k.item():.4f} (true: {k_true})")
print(f"Relative error: {abs(model_inv.k.item() - k_true) / k_true * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# k convergence
axes[0].plot(k_history, 'b-', linewidth=1)
axes[0].axhline(y=k_true, color='r', linestyle='--', label=f'True k = {k_true}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('k')
axes[0].set_title('Learned k Over Training')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Solution comparison
t_fine_t = torch.linspace(0, 2, 200).unsqueeze(1)
with torch.no_grad():
    u_inv_pred = model_inv(t_fine_t).numpy()
u_true = np.exp(-k_true * t_fine_t.numpy())

axes[1].plot(t_fine_t.numpy(), u_true, 'b-', label='True solution', linewidth=2)
axes[1].plot(t_fine_t.numpy(), u_inv_pred, 'r--', label='PINN (learned k)', linewidth=2)
axes[1].scatter(t_obs_np, u_obs_np, c='gray', s=20, alpha=0.5, label='Data')
axes[1].set_xlabel('t'); axes[1].set_ylabel('u(t)')
axes[1].set_title('Inverse PINN: Inferred Solution')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## How It Works

The inverse PINN has **two kinds of trainable parameters**:

1. **Network weights** (thousands) — approximate the solution `u(t)`
2. **Physical parameters** (one: `k`) — appear in the physics loss

Both are optimized simultaneously by the same optimizer. The data loss anchors the solution to observations, and the physics loss forces consistency with the ODE — jointly constraining `k`.

### Why `log_k`?

We parameterize as `k = exp(log_k)` to ensure `k > 0` without constrained optimization. This is common for positive physical parameters.

### Real-World Applications

- **Fluid dynamics**: infer viscosity or Reynolds number from velocity measurements
- **Materials science**: estimate thermal conductivity from temperature data
- **Biology**: find reaction rates from concentration time series
- **Finance**: calibrate model parameters from market data

See `experiments/navier_stokes_inverse/` in this repo for inferring Reynolds number from flow data.

## Exercises

1. **Harder inverse**: Try with true `k = 0.1` (slow decay). Is it harder or easier to infer?
2. **More noise**: Increase noise to 10%. How robust is the inference?
3. **Less data**: Use only 5 observations. When does the inverse problem become ill-posed?
4. **Two parameters**: Modify to `u' = -k*u + b` with unknown `k` and `b`. Can you infer both?

## What's Next

**Notebook 08** gives the honest assessment: when PINNs work, when they fail, what the alternatives are, and a decision framework for choosing the right approach.